# Neural Networks

This PAK aims to introduce you to toy problems with neural networks.

Often you will test the properties of a neural network out first by testing it on some dummy data that you have generated yourself. The goal is to generate the data such that if the network can't solve this toy problem first, then it definitely won't behave reasonably on a real task.


KATE expects your code to define variables with specific names that correspond to certain things we are interested in.

KATE will run your notebook from top to bottom and check the latest value of those variables, so make sure you don't overwrite them.

* Remember to uncomment the line assigning the variable to your answer and don't change the variable or function names.
* Use copies of the original or previous DataFrames to make sure you do not overwrite them by mistake.

You will find instructions below about how to define each variable.

Once you're happy with your code, upload your notebook to KATE to check your feedback.

## Setup
First, we provide some functions that will be useful throughout the exercise.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.datasets import make_blobs, make_moons
from sklearn.model_selection import train_test_split

SEED = 0
np.random.seed(SEED)
tf.random.set_seed(SEED)

def plot_decision_boundary(X, y, model, steps=1000, cmap='Paired'):
    cmap = plt.get_cmap(cmap)

    # Define region of interest by data limits
    xmin, xmax = X[:,0].min() - 1, X[:,0].max() + 1
    ymin, ymax = X[:,1].min() - 1, X[:,1].max() + 1
    
    x_span = np.linspace(xmin, xmax, 100)
    y_span = np.linspace(ymin, ymax, 100)
    
    
    xx, yy = np.meshgrid(x_span, y_span)

    labels = model.predict(np.c_[xx.ravel(), yy.ravel()])
    labels = np.argmax(labels, axis=-1)
    
    z = labels.reshape(xx.shape)

    fig, ax = plt.subplots()
    ax.contourf(xx, yy, z, cmap=cmap, alpha=0.5)

    train_labels = model.predict(X)
    ax.scatter(X[:,0], X[:,1], c=y.ravel(), cmap=cmap, lw=0)

    return fig, ax

## Toy problems

Most of the time when you start a new project with a neural network you're going to reach for a multi-layer perceptron (MLP) as your baseline.

The core building block of the MLP is the *densely-connected* or *linear* layer.

The densely connected layer computes $y = W^\top x$, where $W$ is a weight matrix and $x$ is the input.

In Keras we can create a `Dense` layer as follows:

In [ ]:
MLP_layer = layers.Dense(2, use_bias=False)

x = np.array([[1, 2]]) # make a vector
MLP_layer(x).numpy()[0] # index to remove batch dimension

We can recreate what Keras has done using numpy.

**Q1. To test this we can reproduce the keras layer, define a new variable `our_layer_output`  $W^\top x$, you'll want to use indexing to select the first element of x in order to remove the batch dimension.**

In [ ]:
W = MLP_layer.kernel.numpy()

# Your code here
# our_layer_output = 
#     ...


Once you have defined `our_layer_output`, you should be able to execute the cell below and check that the outputs are the same:

In [ ]:
keras_layer_output = MLP_layer(x).numpy()
print('The Keras layer output is: ', keras_layer_output)
#print('Our layer output is: ' ,our_layer_output)

Imagine we'd like to use our layer to do classification between two classes (i.e. we want the output to be a vector of two probabilities, one for class 1 and another for class 2).

We get the output:

> `[-1.89889419  0.40445614]`

When we run the above cell. (Yours will differ due to the random initialisation of the weights `W`.)

Can you see a problem here?

If we try to interpret these as class probabilities we're going to run into problems, because `-1.89` isn't a valid probability (probabilities must be between 0 and 1). Also, the probability vector should sum to 1.

How can we achieve this?

**Q2. Construct a Keras model with one Dense layer and a second layer which normalises the outputs of the first layer into a probability distribution**

In the cell below, `normaliser` is defined as a ReLU (or rectified linear unit), which leaves positive values unchanged, but sets negative values to 0. We want to redefine this layer so that it outputs a probability distribution (i.e. all values are between 0 and 1 and their sum is equal to 1). It may be useful to read the *keras* documentation.

Having redefined the `normaliser` layer, create a `Sequential` model of the form:

```python
model = tf.keras.Sequential([
    layers.Dense(2, use_bias=False),
    normaliser,
])
```

where the `Dense` layer is followed by another type of layer to perform this probability normalisation. 

In [ ]:
normaliser = layers.ReLU()

# Your code here...
# normaliser = ...
# model = ...


Now when we run the below cell we should get a normalised probability vector (i.e. non-negative and sums to one):

Now let's create a new model, called `multi_layer_model`. Here we can stack multiple of these `Dense` layers together to create a multi-layer neural network. Note that you will need to implement the `normaliser` layer before running this cell:

In [ ]:
multi_layer_model = tf.keras.Sequential([
    layers.Dense(10),
    layers.Dense(10),
    layers.Dense(10),
    layers.Dense(2),
    normaliser,
])

multi_layer_model.build(input_shape=(1,2))

multi_layer_model.summary()

Remember that each of these layers is just doing a matrix-vector product between their weight matrix and an input vector. Layer 1 will output a vector, which is input to layer 2 and so on.

Let's test the network out on a dummy dataset.

In [ ]:
# fix the seed for the random generator
SEED = 123456
np.random.seed(SEED)
tf.random.set_seed(SEED)

# get a randomly generated dataset from sklearn
X, y = make_blobs(centers=2, n_samples=100)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33)

Now that we have defined our train and test data, we can called the `plot_decision_boundary` function:

In [ ]:
plot_decision_boundary(X_train, y_train, multi_layer_model, cmap='RdBu')

The scatter points show you the training examples, and the shaded area shows you the decision boundaries of the neural network. With a random initialisation, our network is pretty bad.

**Q3. Define a function `def fit(model, X, y):`, which compiles the model with a loss and an optimiser, fits the model on `X_train` and `y_train` and returns the trained `model`**

For the optimiser, use `'SGD'`. You will need to think about which type of loss to use! Keras gives you a few options for classification:
- `BinaryCrossentropy`
- `CategoricalCrossentropy`
- `SparseCategoricalCrossentropy`
 
Call your function in the cell below and play with the batch size and number of epochs until you get a loss under ~.3.

Note: don't forget to return the trained model in your function with `return model`

In [ ]:
# Your code here...
def fit(model, X, y):
    # Train the model
    return model


Now if we call your fit function and plot the decision boundary we should get something a little more reasonable:

In [ ]:
multi_layer_model = fit(multi_layer_model, X, y)

We should be able to extract a fairly sensible looking decision boundary:

In [ ]:
plot_decision_boundary(X_train, y_train, multi_layer_model, cmap = 'RdBu')

Now let's try reusing our model on a slightly more complex dummy dataset:

In [ ]:
X, y = make_moons(n_samples=100, noise=.1)
multi_layer_model = fit(multi_layer_model, X, y)
plot_decision_boundary(X, y, multi_layer_model, cmap = 'RdBu')

This isn't bad! But can you see a problem with the decision boundary we are drawing? It's a straight line, and our data is obviously not linearly separable (i.e. it's curvy and can't be separated into classes by a straight line).

Remember that we re-implemented the Keras `Dense` layer and it was just a matrix-vector product of a matrix $W$ and a vector $x$. This is a **linear combination** of the columns of $W$, where the coefficients are the elements of $x$.

This is quite an important point, so make sure you're happy with it. The phrase **linear combination** in particular should give us a hint! By stacking up `Dense` layers, we are really just doing a sequence of linear transformations of the input - this means our network can only express linear functions. In fact, at the moment, we could just multiply all of our matrices together and reduce all of our layers down into just a single layer.

We need to add something non-linear into the network in order to allow ourselves to express non-linearly-separable data. In general, we're going to want to put something non-linear after every layer (otherwise we could just multiply the layers together).

**Q4. Redefine the model, adding non-linear activation functions after each of the model layers. Play with different activation functions and see which works best.**

Name your model `model_with_activations`.

In [ ]:
# Your code here
# model_with_activations = ...
# model_with_activations.build(input_shape=(1,2))
# fit(model_with_activations, X, y)


Having implemented the above model and trained it, we can see that when you plot the decision boundary it will be non-linear:

In [ ]:
#plot_decision_boundary(X, y, model_with_activations, cmap='RdBu')

Hopefully this serves as a good illustration of the use case for toy problems, especially when it comes to trying to discover limitations of your networks. 